In [ ]:
# PPR annotation pipeline
# note that all 'files' are contained in quotes, so adjust these for your file locations
# %pip install pyranges

In [1]:
# packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Bio import SeqIO, AlignIO
from Bio.Seq import Seq
import re
import os
import pyranges as pr

In [ ]:
# fna, run fasta_df before
fna = fasta_df('/home/pmh3ax/start_files/S.latifolia_v5.0.genomeanchored.fna')
fna['Headers']

In [ ]:
fna = fna.loc[~fna['Headers'].str.contains('scaffold', na=False)]
fna['Headers']

In [ ]:
fna['unique_chars'] = fna['Sequences'].apply(lambda x: ''.join(sorted(set(x))))
fna['unique_chars']

In [19]:
# export
def df_to_fasta(df, outputfile):
    with open(outputfile, 'w') as file:
        for index, row in df.iterrows():
            header = row['Headers']
            if not header.startswith('>'):
                header = '>' + header
            file.write(f"{header}\n{row['Sequences']}\n")
# df_to_fasta(fna, '/home/pmh3ax/pprfinder_runs/l_run/Sl5.fa') - uncomment when actual run

In [ ]:
# index, module load before
samtools faidx '/home/pmh3ax/pprfinder_runs/l_run/Sl5.fa'

In [ ]:
# run in command line
grep -E 'ID' '/home/pmh3ax/start_files/Slatifolia.v5.complete.gff' > '/home/pmh3ax/start_files/Sl5.gff'
grep 'chr' '/home/pmh3ax/start_files/Sl5.gff' > '/home/pmh3ax/pprfinder_runs/l_run/Sl5.gff'

# note that this can also be used to filter for specific annotation types before downstream process
# to do this, replace ID with cds, gene, exon, mrna, or another type
# for latifolia also replace with 'chr' to avoid scaffold_ annotations

In [ ]:
# df process
gff = pd.read_csv('/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type', 'start', 'end', '.', 'strand', 'score', 'ID']

In [ ]:
def replace_dots(match):
    return match.group(0).replace('.', '_')
gff['ID'] = gff['ID'].str.replace(r'ID=[^;]+', replace_dots, regex=True)
gff['ID']

In [ ]:
gff = gff.loc[~gff['chr'].str.contains('scaffold', na=False)]
gff['chr'].value_counts()

In [ ]:
gff = gff.loc[~gff['type'].str.contains('intergenic', na=False)]
gff['type'].value_counts()

In [ ]:
def read_fai(fai_file):
    chr_lengths = {}
    with open(fai_file, 'r') as file:
        for line in file:
            parts = line.strip().split('\t')
            chr_lengths[parts[0]] = int(parts[1])
    return chr_lengths

def filter_gff_by_fai(gff_df, fai_file):
    chr_lengths = read_fai(fai_file)

    missing_chromosomes = set(gff_df['chr']) - set(chr_lengths.keys())
    if missing_chromosomes:
        raise ValueError(f"Missing chromosomes in FAI file: {', '.join(missing_chromosomes)}")

    gff_df_filtered = gff_df[gff_df['end'] <= gff_df['chr'].map(chr_lengths).fillna(float('inf'))]
    return gff_df_filtered

In [ ]:
gff = filter_gff_by_fai(gff, '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5.fa.fai')
gff['chr'].value_counts()

In [ ]:
# save
gff.to_csv('/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5_fix.gff', sep='\t', index=False, header=None)

In [ ]:
gff = pd.read_csv('/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5_fix.gff', sep='\t', header=None)
gff

In [ ]:
# gffread run in command line cd
./gffread -w '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5exon.fasta' -x '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5cds.fasta' -y '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5translatecds.fasta' -g '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5.fa' -l 279 -u -S '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5_fix.gff'

In [ ]:
trans

In [ ]:
input_fasta = '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5translatecds.fasta'
output_fasta = '/home/pmh3ax/pprfinder_runs/l_run/Sl5transcds.fasta'

with open(output_fasta, "w") as out_handle:
    for record in SeqIO.parse(input_fasta, "fasta"):
        record.seq = Seq(str(record.seq).replace(".", "*"))  # Convert Seq -> str, modify, back to Seq
        SeqIO.write(record, out_handle, "fasta")

print(f"Cleaned FASTA saved to {output_fasta}")

In [ ]:
# after gffread and before hmmer, translate the cds- translatecds from gffread was buggy

In [ ]:
# run in command line- hmmer search
export PATH='/home/pmh3ax/hmmer-3.4/src':$PATH
# note pprs.domt is not used downstream
# all_CTD_Smr is the hmm profile provided by Small
hmmsearch --noali -E 0.1 -o pprs.domt --domtblout '/home/pmh3ax/pprfinder_runs/l_run/Sl5_TabOut' '/home/pmh3ax/start_files/all_CTD_Smr.hmm' '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5transcds.fasta'

In [ ]:
# run in command line- julia pprfinder
module load julia
julia
] # enter package manager
add BioSequences FASTX
# backspace to exit package manager
exit()
julia ./PPRfinder/PPRfinder.jl '/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5transcds.fasta' '/home/pmh3ax/pprfinder_runs/l_run/Sl5_TabOut'

In [ ]:

# for loading PPR annotations onto gff


In [2]:
def fasta_df(inputfile):
    headers = []
    sequences = []
    currentsequence = ''
    with open(inputfile, 'r') as file:
        for line in file.readlines():
            line = line.strip()
            if line.startswith('>'):
                if currentsequence:
                    sequences.append(currentsequence)
                    currentsequence = ''
                headers.append(line)
            else:
                currentsequence += line
        if currentsequence:
            sequences.append(currentsequence)
    df = pd.DataFrame({'Headers': headers, 'Sequences': sequences})
    return df

In [ ]:
# pprs fasta result
fasta = fasta_df('/home/pmh3ax/pprfinder_runs/l_run/Sl5_TabOut.pprs.fa')
fasta
# line below gets rid > at beginning, only run once
fasta['Headers'] = fasta['Headers'].str.slice(start=1)
string_list = fasta['Headers'].tolist()

In [ ]:
# gff to match, will work with pprfinder output since the ID and type columns contain matching IDs
gff = pd.read_csv('/home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5_fix.gff', sep='\t', header=None)
gff.columns = ['Chromosome', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
gff

In [ ]:
gff['type'].value_counts()

In [ ]:
# match gff annotations with pprs defined by pprfinder output
# False, non ppr = 0
# True, ppr = 1
# got rid of all gene, need to add an exception
gff['PPR'] = gff['ID'].apply(lambda x: any(header in x for header in string_list))
gffppr = gff[gff['PPR'] == True]
gffppr.drop('PPR', axis=1)

In [ ]:
gffppr['type'].value_counts()

In [ ]:
# pull up here <-- ignore
gffppr.to_csv('/home/pmh3ax/pprfinder_runs/l_run/Sl5_gffppr.gff', sep='\t', index=False, header=None)

In [ ]:
gff_genes = gff[gff['type'] == 'mRNA']

pr_ppr = pr.PyRanges(gffppr[['Chromosome', 'Start', 'End']])
pr_genes = pr.PyRanges(gff_genes[['Chromosome', 'Start', 'End']])

overlapping = pr_genes.overlap(pr_ppr)
overlapping_genes = gff_genes.merge(overlapping.df, on=['Chromosome', 'Start', 'End'], how='inner')

gffppr_filtered = pd.concat([gffppr, overlapping_genes]).drop_duplicates()

In [ ]:
gffppr_filtered['type'].value_counts()

In [ ]:
# save to gff
gffppr_filtered.to_csv('/home/pmh3ax/pprfinder_runs/l_run/Sl5pprsgene.gff', sep='\t', index=False, header=None)

In [ ]:

# before the below step, manually remove unreliable annotations (see notes)
# import as separate tsv/csv/other


In [ ]:
# or if continuous no load, did not manually check
pprs = gffppr
pprs.drop('PPR', axis=1)

In [ ]:
# ppr gff
pprs = pd.read_csv('/home/pmh3ax/pprfinder_runs/l_run/Sl5pprsgene.gff', sep='\t', header=None)
pprs.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID', 'ppr']
# all pprs, safe to drop col
pprs = pprs.drop('ppr', axis=1)

In [ ]:
# excludes utrs, not ppr significant, generally
pprs['type'].value_counts()
pprs = pprs[~pprs['type'].isin(['three_prime_UTR', 'five_prime_UTR'])]

In [ ]:
# manual annotation reduced
annot = pd.read_csv('/home/pmh3ax/pprfinder_runs/l_run/S.latifolia_v5.0pprs_genecds.tsv', sep='\t', header=None)
# if different, note and adjust accordingly for match, bounds
annot.columns = ['name', 'chr', 'type', 'Start', 'End', 'length', 'direction']

In [ ]:
# name column in the extracted annotation tsv is simply type mrna, exon, etc
annot = annot.drop(0)
annot['type'].value_counts()

In [ ]:
# more filter
annot = annot[annot['type']!='CDS']
annot['length'] = annot['length'].astype(int)
# spurious remove
annot = annot[annot['length']>2]

In [ ]:
# merge pprs gff with annot manual by using same annotation bounds- check same count
annot['Start'] = annot['Start'].astype(int)
annot['End'] = annot['End'].astype(int)
pprs_manual = pprs.merge(annot[['chr', 'Start', 'End']], on=['chr', 'Start', 'End'], how='inner')

In [ ]:
# perhaps a few dupes still remain after manual
pprs_manual = pprs_manual.drop_duplicates()

In [ ]:
annot['type'].value_counts()

In [ ]:
pprs_manual = pprs_manual[pprs_manual['type']!='exon']
pprs_manual['type'].value_counts()

In [ ]:
# export all genes
pprs_manual.to_csv('/home/pmh3ax/pprfinder_runs/l_run/Sl5pprs_manual.gff', sep='\t', index=False, header=None)

In [ ]:
# read in pprs_manual
pprs_manual = pd.read_csv('/home/pmh3ax/pprfinder_runs/l_run/Sl5pprs_manual.gff', sep='\t', header=None)
pprs_manual.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
pprs_manual

In [ ]:
# motifs structure import (whole protein) to find reliable repeat genes
beads = pd.read_csv('/home/pmh3ax/pprfinder_runs/l_run/Sl5_TabOut.beads.txt', sep='\t', header=None)
beads.columns = ['ID', 'length', 'motif_arrangement', 'type', 'score']

In [ ]:
pprs_manual

In [ ]:
beads

In [ ]:
# count column, then filter 3 (each motif 1 rna nt, 1 or 2 motifs not reliable repeat)
beads['motif_count'] = beads['motif_arrangement'].apply(lambda x: sum(c.isalpha() for c in str(x)))
beads = beads[beads['motif_count']>=3]

In [ ]:
beads['motif_clean'] = beads['motif_arrangement'].apply(lambda x: re.sub(r'[^a-zA-Z]', '', x))
beads["ID"] = beads["ID"].str.replace(r"\.\d+$", "", regex=True)

In [ ]:
# beads with pprs_manual by clean id without unnecessary info, then filter to genes
pprs_manual['ID_clean'] = pprs_manual['ID'].str.extract(r'ID=([^;]+)')
pprs_gene = pprs_manual[pprs_manual['type']=='gene']

In [ ]:
# intersection, only beads
pprs_gene_filtered = pprs_gene[pprs_gene['ID_clean'].isin(beads['ID'])].copy()

In [ ]:
pprs_gene_filtered

In [ ]:
beadsrfl = beads[beads['motif_clean'].str.contains('RFLCTD')]
beadsrfl

In [ ]:
# scratch to determine target
# pprs_gene['ID_clean'].unique() #print all
# find which do not have scaffold_n, target string: FUN
# copy
pprs_gene_name = pprs_gene_filtered.copy()

In [ ]:
pprs_gene_filtered

In [ ]:
pprs_gene_filtered = pprs_gene_filtered[pprs_gene_filtered['ID_clean'].isin(beadsrfl['ID'])]

In [ ]:
pprs_gene_filtered = pprs_gene_filtered.drop(columns='ID_clean', axis=1)

In [ ]:
# standardize names that do not include scaffold- not needed do not run
target = 'Chr'
for index, row in pprs_gene_name.iterrows():
    if target in row['ID_clean']:
        row_old = row['ID_clean']
        pprs_gene_name.at[index, 'ID_clean'] = 'Silene_latifolia_' + row['chr'] + '_' + row_old
pprs_gene_name = pprs_gene_name.drop(columns='ID', axis=1)

In [ ]:
# pprs_gene_name = pprs_gene_name.drop('ID', axis=1)
# add to export pprs_gene
# changed path so would not change original in l_run
pprs_gene_name.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5_pprs_clean.gff', sep='\t', index=False, header=None)
pprs_gene_filtered.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5_pprsrfl_clean.gff', sep='\t', index=False, header=None)

In [ ]:
# export beads for grouping
beads.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5_beads_clean.gff', sep='\t', index=False, header=None)

In [ ]:
# or, read in here picking up
# CHANGE FOR SPECIFIC SPECIES

In [ ]:
pprs_gene_name

In [ ]:
# motif sort
beadsp = beads[beads['type']=='P']
beadsp_only = beadsp[~beadsp['motif_clean'].str.contains('RFLCTD')]
beadsp_rfl = beadsp[beadsp['motif_clean'].str.contains('RFLCTD')]

In [ ]:
# sort
beadsp_onlys = beadsp_only.sort_values(by='motif_count', ascending=False)
beadsp_rfls = beadsp_rfl.sort_values(by='motif_count',ascending=False)

In [ ]:
# export
beadsp_onlys.to_csv('/home/pmh3ax/pprfinder_runs/l_run/sl5_p_only.gff', sep='\t', index=False, header=None)
beadsp_rfls.to_csv('/home/pmh3ax/pprfinder_runs/l_run/sl5_p_rfl.gff', sep='\t', index=False, header=None)

In [ ]:
# pls
beadspls = beads[beads['type']=='PLS']
beadspls_e = beadspls[beadspls['motif_clean'].str.contains('E') & ~beadspls['motif_clean'].str.contains('DYW')]
beadspls_edyw = beadspls[beadspls['motif_clean'].str.contains('DYW') & beadspls['motif_clean'].str.contains('E')]
beadspls_smr = beadspls[beadspls['motif_clean'].str.contains('Smr')]
beadspls_only = beadspls[~beadspls['motif_clean'].str.contains('DYW') & ~beadspls['motif_clean'].str.contains('E') & ~beadspls['motif_clean'].str.contains('Smr')]

In [ ]:
beadspls_only['motif_clean'].value_counts()

In [ ]:
beadspls_e['motif_clean'].value_counts()

In [ ]:
beadspls_edyw['motif_clean'].value_counts()

In [ ]:
beadspls_smr['motif_clean'].value_counts()

In [ ]:
# probably fine to keep with 5 rows, should not have more than this many classes in general
beadspls_onlys = beadspls_only.sort_values(by='motif_count', ascending=False)
beadspls_es = beadspls_e.sort_values(by='motif_count', ascending=False)
beadspls_edyws = beadspls_edyw.sort_values(by='motif_count', ascending=False)
beadspls_smrs = beadspls_smr.sort_values(by='motif_count', ascending=False)

In [ ]:
# export
beadspls_onlys.to_csv('/home/pmh3ax/pprfinder_runs/l_run/sl5_pls_only.gff', sep='\t', index=False, header=None)
beadspls_es.to_csv('/home/pmh3ax/pprfinder_runs/l_run/sl5_pls_e.gff', sep='\t', index=False, header=None)
beadspls_edyws.to_csv('/home/pmh3ax/pprfinder_runs/l_run/sl5_pls_edyw.gff', sep='\t', index=False, header=None)
beadspls_smrs.to_csv('/home/pmh3ax/pprfinder_runs/l_run/sl5_pls_smr.gff', sep='\t', index=False, header=None)

In [ ]:
pprs_gene_filtered

In [ ]:
pprs_gene_name

In [ ]:
pprs_gene

In [ ]:
def beads_to_pprs_gene(beadsgff, pprs_gene, idname):
    # Filter pprs_gene based on beadsgff IDs
    pprs_gene_filtered = pprs_gene[pprs_gene['ID_clean'].isin(beadsgff['ID'])].copy()
    pprs_gene_filtered['ID_clean'] = 'Silene_latifolia_' + pprs_gene_filtered['ID_clean']
    if 'ID_clean' in pprs_gene_filtered.columns:
        pprs_gene_filtered = pprs_gene_filtered.drop(columns=['ID_clean'])
    output_path = os.path.join('/home/pmh3ax/pprfinder_runs/l_run', f'Sl5_pprs_clean_{idname}.gff')
    pprs_gene_filtered.to_csv(output_path, sep='\t', index=False, header=None)

In [ ]:
beads_to_pprs_gene(beadsp_onlys, pprs_gene, 'p_only')
beads_to_pprs_gene(beadsp_rfls, pprs_gene, 'p_rfl')

In [ ]:
beads_to_pprs_gene(beadspls_onlys, pprs_gene, 'pls_only')
beads_to_pprs_gene(beadspls_smrs, pprs_gene, 'pls_smr')
beads_to_pprs_gene(beadspls_es, pprs_gene, 'pls_e')
beads_to_pprs_gene(beadspls_edyws, pprs_gene, 'pls_edyw')

In [ ]:
# automated fix id
awk 'BEGIN {OFS="\t"} 
    $3 == "gene" { $9 = "ID=" $9 } 
    { print }' Sl5_pprs_clean_p_only.gff > Sl5_pprs_clean_p_onlyf.gff

In [ ]:
# extract transcripts from pprs, modify each in command line # CHANGE FOR SPECIFIC SPECIES
# cd gffread
# if the slighly processed fa doesn't work, then go to og start file one
./gffread -w /home/pmh3ax/pprfinder_runs/l_run/p_only.fa -g /home/pmh3ax/start_files/S.latifolia_v5.0.genomeanchored.fna /home/pmh3ax/pprfinder_runs/l_run/Sl5_pprs_clean_p_only.gff
./gffread -w /home/pmh3ax/pprfinder_runs/l_run/p_rfl.fa -g /home/pmh3ax/start_files/S.latifolia_v5.0.genomeanchored.fna /home/pmh3ax/pprfinder_runs/l_run/Sl5_pprs_clean_p_rfl.gff
./gffread -w /home/pmh3ax/pprfinder_runs/l_run/pls_only.fa -g /home/pmh3ax/start_files/S.latifolia_v5.0.genomeanchored.fna /home/pmh3ax/pprfinder_runs/l_run/Sl5_pprs_clean_pls_only.gff
./gffread -w /home/pmh3ax/pprfinder_runs/l_run/pls_smr.fa -g /home/pmh3ax/start_files/S.latifolia_v5.0.genomeanchored.fna /home/pmh3ax/pprfinder_runs/l_run/Sl5_pprs_clean_pls_smr.gff
./gffread -w /home/pmh3ax/pprfinder_runs/l_run/pls_e.fa -g /home/pmh3ax/start_files/S.latifolia_v5.0.genomeanchored.fna /home/pmh3ax/pprfinder_runs/l_run/Sl5_pprs_clean_pls_e.gff
./gffread -w /home/pmh3ax/pprfinder_runs/l_run/pls_edyw.fa -g /home/pmh3ax/start_files/S.latifolia_v5.0.genomeanchored.fna /home/pmh3ax/pprfinder_runs/l_run/Sl5_pprs_clean_pls_edyw.gff



In [ ]:
# align each, then cat
# then run in mafft -merge (realign)
# trim trimal
# tree IQTree bootstrap

In [ ]:
# loading rfl group
beads = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sl5_beads_clean.gff', sep='\t', header=None)
gff = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sl5_pprsrfl_clean.gff', sep='\t', header=None)

In [ ]:
# get rid of few beads 
beads[7] = beads[0].apply(lambda x: gff[8].str.contains(x, na=False).any())
extra_rows = beads[~beads[7]]

In [ ]:
beadsf = beads[beads[7]].drop(columns=[7]).reset_index(drop=True)

In [ ]:
# all beads in gff
extra_rows_check = beadsf[~beadsf[0].apply(lambda x: gff[8].str.contains(x, na=False).any())]
extra_rows_check

In [ ]:
def partial_merge(df1, df2, key_col, full_key_col):
    df1['match'] = df1[key_col].apply(lambda x: df2[df2[full_key_col].str.contains(x, na=False, regex=False)][full_key_col].tolist())
    return df1.explode('match').merge(df2, left_on='match', right_on=full_key_col, how='left').drop(columns=['match'])

# Perform the merge with correct column names
merged_gff = partial_merge(beads, gff, key_col=0, full_key_col=8)

In [ ]:
# Drop unwanted columns
merged_gff.drop(columns=['0_x', '1_x', '3_x', '4_x', '5_x', '7_x'], inplace=True)
remaining_columns = merged_gff.columns.tolist()
new_order = [col for col in remaining_columns if col not in ['2_x', '6_x']] + ['2_x', '6_x']
merged_gff = merged_gff[new_order].dropna()
merged_gff

In [ ]:
merged_gff.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5_pprs_beads_rfl.gff', sep='\t', header=None, index=False)

In [ ]:
# put on geneious chr view..
# then get allpprs and rf gff for fasta convert

In [ ]:
# Read the input GFF file
input_file = '/home/pmh3ax/pprs_analysis_other/Sl5_pprs_clean.gff'
output_file = '/home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_allpprs.gff'
input_file2 = '/home/pmh3ax/pprs_analysis_other/Sl5_pprsrfl_clean.gff'
output_file2 = '/home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_rfl.gff'

In [ ]:
with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        if line.strip():  # Check if the line is not empty
            columns = line.strip().split('\t')
            # Add ID= before the last column
            columns[2] = 'mRNA'
            columns[-1] = f'ID={columns[-1]}'
            # Write the modified line to the output file
            outfile.write('\t'.join(columns) + '\n')

In [ ]:
with open(input_file2, 'r') as infile, open(output_file2, 'w') as outfile:
    for line in infile:
        if line.strip():  # Check if the line is not empty
            columns = line.strip().split('\t')
            # Add ID= before the last column
            columns[-1] = f'ID={columns[-1]}'
            # Write the modified line to the output file
            outfile.write('\t'.join(columns) + '\n')

In [ ]:
# to get columns for each, rfl/non
mastergff = pd.read_csv(input_file, sep='\t', header=None)
mastergff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
mastergff

In [ ]:
rflgff = pd.read_csv(input_file2, sep='\t', header=None)
rflgff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
rflgff

In [ ]:
mastergff['rfl'] = mastergff['ID'].apply(lambda x: rflgff['ID'].str.contains(x).any())
mastergff['rfl'].value_counts()

In [ ]:
mastergff['type'] = 'mRNA'
mastergff

In [ ]:
rflgff['type'] = 'mRNA'
rflgff

In [ ]:
nonrflgff = mastergff[mastergff['rfl']==False]
nonrflgff = nonrflgff.drop(columns='rfl', axis=1)
nonrflgff

In [ ]:
rflgff.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5_pprs_clean_rfl.gff', sep='\t', header=None, index=False)

In [ ]:
nonrflgff.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5_pprs_clean_nonrfl.gff', sep='\t', header=None, index=False)

In [ ]:
input_file3 = '/home/pmh3ax/pprs_analysis_other/Sl5_pprs_clean_nonrfl.gff'
output_file3 = '/home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_nonrfl.gff'

In [ ]:
with open(input_file3, 'r') as infile, open(output_file3, 'w') as outfile:
    for line in infile:
        if line.strip():  # Check if the line is not empty
            columns = line.strip().split('\t')
            # Add ID= before the last column
            columns[-1] = f'ID={columns[-1]}'
            # Write the modified line to the output file
            outfile.write('\t'.join(columns) + '\n')

In [ ]:
# gffread to get transcripts sequences
./gffread -w /home/pmh3ax/pprs_analysis_other/Sl5rflseqs.fa -g /home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5.fa /home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_rfl.gff
./gffread -w /home/pmh3ax/pprs_analysis_other/Sl5allpprsseqs.fa -g /home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5.fa /home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_allpprs.gff
./gffread -w /home/pmh3ax/pprs_analysis_other/Sl5nonrflseqs.fa -g /home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5.fa /home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_nonrfl.gff


In [ ]:
# might have to manually fix a few of these

In [ ]:
# praise the LORD this worked

In [ ]:
# build hmm- requires biopython convert to change afa to sto
AlignIO.convert('/home/pmh3ax/pprs_analysis_other/Sl5rflseqs-alignment.fasta', "fasta",
                '/home/pmh3ax/pprs_analysis_other/Sl5rflseqs_hmm.sto', "stockholm")

In [ ]:
# hmm path thing
export PATH='/home/pmh3ax/hmmer-3.4/src':$PATH
# build rfl profile
hmmbuild /home/pmh3ax/pprs_analysis_other/Sl5rflseqs.hmm /home/pmh3ax/pprs_analysis_other/Sl5rflseqs_hmm.sto

In [ ]:
# run hmmer
# path if not already
nhmmer --noali -E 0.1 -o pprs.domt --tblout /home/pmh3ax/pprs_analysis_other/Sl5nonrflseqsTabOut /home/pmh3ax/pprs_analysis_other/Sl5rflseqs.hmm /home/pmh3ax/pprs_analysis_other/Sl5nonrflseqs.fa

In [ ]:
# intersecting hits from nt and aa hmmsearch- do not need to run
nthmmresult = "/home/pmh3ax/pprs_analysis_other/Sl5nonrflseqsTabOut"
aahmmresult = "/home/pmh3ax/pprs_analysis_other/Sl5nonrflseqs-translationTabOut"
df_nthmm = pd.read_csv(nthmmresult, sep='\s+', header=None)
df_aahmm = pd.read_csv(aahmmresult, sep='\s+', header=None)
df_aahmm[0] = df_aahmm[0].str.replace("_translation", "", regex=False)

# Find common IDs in the first column
common_ids = set(df_nthmm[0]) & set(df_aahmm[0])
df_nthmm_filtered = df_nthmm[df_nthmm[0].isin(common_ids)]
df_aahmm_filtered = df_aahmm[df_aahmm[0].isin(common_ids)]

df_nthmm_filtered.to_csv(nthmmresult + "_filtered", sep=" ", index=False, header=False)
df_aahmm_filtered.to_csv(aahmmresult + "_filtered", sep=" ", index=False, header=False)
print("Filtering complete. Saved filtered files.")

In [ ]:
# match with gff to id seqs
nthmm_filtered = "/home/pmh3ax/pprs_analysis_other/Sl5nonrflseqsTabOut"
gff_file = "/home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_nonrfl.gff"
output_gff = "/home/pmh3ax/pprs_analysis_other/hmm_id_nonrfl.gff"

df_nthmm = pd.read_csv(nthmm_filtered, sep='\s+', header=None)
valid_ids = set(df_nthmm[0])

gff = pd.read_csv(gff_file, sep="\t", header=None, comment='#', dtype=str)

def id_in_gff(attributes):
    return any(id_ in attributes for id_ in valid_ids)

gff_filtered = gff[gff.iloc[:, 8].apply(id_in_gff)]

# Save filtered GFF
gff_filtered.to_csv(output_gff, sep="\t", index=False, header=False)

print("GFF filtering complete. Saved to:", output_gff)

In [ ]:
# fasta
./gffread -w /home/pmh3ax/pprs_analysis_other/Sl5nonrfl_hmm_seqs.fa -g /home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5.fa /home/pmh3ax/pprs_analysis_other/hmm_id_nonrfl.gff

In [ ]:
input_file = "/home/pmh3ax/pprs_analysis_other/Sl5rflseqs-translation.fasta"  # Change this to your filename
output_file = "/home/pmh3ax/pprs_analysis_other/Sl5rflseqs-translation_fixed.fasta"

with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        if line.startswith(">"):
            # Remove the specified prefix
            line = re.sub(r'^>Silene_latifolia_Chr\d{2}_', '>', line)
            # Truncate to 50 characters
            line = line[:50]
        outfile.write(line)

In [ ]:
input_file2 = "/home/pmh3ax/pprs_analysis_other/Sl5rflseqs.fa"  # Change this to your filename
output_file2 = "/home/pmh3ax/pprs_analysis_other/Sl5rflseqs_fixed.fa"

with open(input_file2, "r") as infile, open(output_file2, "w") as outfile:
    for line in infile:
        if line.startswith(">"):
            # Remove the specified prefix
            line = re.sub(r'^>Silene_latifolia_Chr\d{2}_', '>', line)
            # Truncate to 50 characters
            line = line[:50]
        outfile.write(line)

In [ ]:
# make blast db, in command line
# module load blast
makeblastdb -in /home/pmh3ax/pprs_analysis_other/blastdb/Sl5rflseqs.fa -dbtype nucl -parse_seqids


In [ ]:
# module if not already
blastn -query /home/pmh3ax/pprs_analysis_other/Sl5nonrflseqs.fa -db /home/pmh3ax/pprs_analysis_other/blastdb/Sl5rflseqs.fa -out /home/pmh3ax/pprs_analysis_other/Sl5nonrflseqs-blast.output -outfmt 6


In [ ]:
# blast results
blast_resultsnt = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sl5nonrflseqs-blast.output', sep='\t', header=None)

In [ ]:
blast_resultsnt1 = blast_resultsnt[blast_resultsnt[2]>=90]
blast_resultsnt1

In [ ]:
# blast results
blast_resultsaa = pd.read_csv('/home/pmh3ax/pprs_analysis_other/blastdb/Sl5nonrflseqs-translationblast.output', sep='\t', header=None)
blast_resultsaa

In [ ]:
blast_resultsaa1 = blast_resultsaa[blast_resultsaa[2]>=85]
blast_resultsaa1

In [ ]:
# blast_resultsnt1
# blast_resultsaa1

blast_resultsaa1[0] = blast_resultsaa1[0].str.replace("_translation", "", regex=False)
common_ids = set(blast_resultsnt1[0]) & set(blast_resultsaa1[0])

blast_resultsnt1_f = blast_resultsnt1[blast_resultsnt1[0].isin(common_ids)]
blast_resultsaa1_f = blast_resultsaa1[blast_resultsaa1[0].isin(common_ids)]

blast_resultsnt1_f.to_csv('/home/pmh3ax/pprs_analysis_other/blastdb/Sl5nonrflseqs-blast.output_filtered.txt', sep="\t", index=False, header=False)
blast_resultsaa1_f.to_csv('/home/pmh3ax/pprs_analysis_other/blastdb/Sl5nonrflseqs-translationblast.output_filtered.txt', sep="\t", index=False, header=False)
print("Filtering complete. Saved filtered files.")

In [ ]:
blast_resultsnt1_f_path = '/home/pmh3ax/pprs_analysis_other/Sl5nonrflseqs-blast.output'
gff_file = '/home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_nonrfl.gff'
output_gff = '/home/pmh3ax/pprs_analysis_other/Sl5blast_id_nonrfl.gff'

valid_ids = set()
with open(blast_resultsnt1_f_path, 'r') as blast_file:
    for line in blast_file:
        fields = line.strip().split("\t")
        if fields:
            valid_ids.add(fields[0])

gff = pd.read_csv(gff_file, sep="\t", header=None, comment='#', dtype=str)

if gff.shape[1] > 8:
    gff_filtered = gff[gff.iloc[:, 8].str.contains('|'.join(valid_ids), na=False)]

gff_filtered.to_csv(output_gff, sep="\t", index=False, header=False)
print("Filtering complete. Saved filtered files.")

In [ ]:
# fasta
./gffread -w /home/pmh3ax/pprs_analysis_other/Sl5nonrfl_blast_seqs.fa -g /home/pmh3ax/pprfinder_runs/l_run/pre-pprfinder/Sl5.fa /home/pmh3ax/pprs_analysis_other/Sl5blast_id_nonrfl.gff

In [ ]:
# File paths
blast_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sl5nonrfl_blast_seqs.fa'
hmm_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sl5nonrfl_hmm_seqs.fa'
combined_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sl5combined_nonrfl_seqs.fa'

blast_records = SeqIO.to_dict(SeqIO.parse(blast_fasta_path, 'fasta'))
hmm_records = SeqIO.to_dict(SeqIO.parse(hmm_fasta_path, 'fasta'))
combined_records = {**blast_records, **hmm_records}

with open(combined_fasta_path, 'w') as output_handle:
    SeqIO.write(combined_records.values(), output_handle, 'fasta')
print("Combined FASTA file saved to:", combined_fasta_path)

In [ ]:
# File paths- coopt to get all rfls potentially
blast_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sl5combined_nonrfl_seqs.fa'
hmm_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sl5rflseqs.fa'
combined_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sl5combined_rfl_seqs.fa'

blast_records = SeqIO.to_dict(SeqIO.parse(blast_fasta_path, 'fasta'))
hmm_records = SeqIO.to_dict(SeqIO.parse(hmm_fasta_path, 'fasta'))
combined_records = {**blast_records, **hmm_records}

with open(combined_fasta_path, 'w') as output_handle:
    SeqIO.write(combined_records.values(), output_handle, 'fasta')
print("Combined FASTA file saved to:", combined_fasta_path)

In [ ]:
# beads and rfl gff
# combined rfl pprs fasta result
fasta = fasta_df('/home/pmh3ax/pprs_analysis_other/Sl5combined_rfl_seqs.fa')
fasta
# line below gets rid > at beginning, only run once
fasta['Headers'] = fasta['Headers'].str.slice(start=1)
string_list = fasta['Headers'].tolist()

In [ ]:
# gff to match, will work with pprfinder output since the ID and type columns contain matching IDs
gff = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sl5_pprs_clean.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
gff

In [ ]:
# match gff annotations with pprs defined by pprfinder output
# False, non ppr = 0
# True, ppr = 1
gff['PPR'] = gff['ID'].apply(lambda x: any(header in x for header in string_list))
gffppr = gff[gff['PPR']==True]
gffppr.drop('PPR', axis=1)

In [ ]:
# save to gff
gffppr.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5combined_rfl_pprs.gff', sep='\t', index=False, header=None)

In [ ]:
beads = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sl5_beads_clean.gff', sep='\t', header=None)
beads

In [ ]:
def partial_merge(df1, df2, key_col, full_key_col):
    df1['match'] = df1[key_col].apply(lambda x: df2[df2[full_key_col].str.contains(x, na=False, regex=False)][full_key_col].tolist())
    return df1.explode('match').merge(df2, left_on='match', right_on=full_key_col, how='left').drop(columns=['match'])

# Perform the merge with correct column names
merged_gff = partial_merge(beads, gffppr, key_col=0, full_key_col='ID')

In [ ]:
# Drop unwanted columns
merged_gff.drop(columns=[0, 1, 3, 4, 5], inplace=True)
remaining_columns = merged_gff.columns.tolist()
new_order = [col for col in remaining_columns if col not in [2, 6]] + [2, 6]
merged_gff = merged_gff[new_order]
merged_gff = merged_gff.dropna(ignore_index=True)
merged_gff

In [ ]:
merged_gff = merged_gff.rename(columns={2: 'motifs', 6: 'motif_clean'})
merged_gff

In [ ]:
merged_gff.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5beads_combined_rfl.gff', sep='\t', header=None)

In [ ]:
merged_gff['motif_clean'].value_counts()
# mostly P, some RFLCTD, PLS

In [ ]:
# come back after rfl tree
blast_fasta_path = '/home/pmh3ax/pprs_analysis_other/rfl-tree/Sl5collapse_rfl_seqs.txt'
hmm_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sl5combined_rfl_seqs.fa'
combined_fasta_path = '/home/pmh3ax/pprs_analysis_other/Sl5condensed_rfl_seqs.fa'

with open(blast_fasta_path) as f:
    target_ids = set(line.strip() for line in f)

hmm_records = SeqIO.to_dict(SeqIO.parse(hmm_fasta_path, 'fasta'))
filtered_records = [hmm_records[seq_id] for seq_id in target_ids if seq_id in hmm_records]

with open(combined_fasta_path, 'w') as output_handle:
    SeqIO.write(filtered_records, output_handle, 'fasta')

print(f"Filtered {len(filtered_records)} unique sequences from full-length data.")
print("Filtered FASTA file saved to:", combined_fasta_path)

In [ ]:
# run tree build on combined seqs

In [ ]:
file1 = "/home/pmh3ax/pprs_analysis_other/rfl-tree/Sl5trim_rfl_seqs.fa"
file2 = "/home/pmh3ax/pprs_analysis_other/Sl5rflseqs.fa"
tree_file = "/home/pmh3ax/pprs_analysis_other/rfl-tree/Sl5trim_rfl_seqs.fa.contree"

# Read headers from both FASTA files
headers1 = {record.id for record in SeqIO.parse(file1, "fasta")}
headers2 = {record.id for record in SeqIO.parse(file2, "fasta")}
# Find common headers
common_headers = headers1 & headers2
# Dictionary to store original -> modified header mappings
header_mapping = {}

def modify_header(record):
    original_id = record.id
    if original_id in common_headers and original_id in headers1:
        modified_id = original_id.replace("_chr", "RFL_chr")
        record.id = modified_id
        record.description = modified_id
        header_mapping[original_id] = modified_id
    return record

# Modify and save updated FASTA file
records1 = (modify_header(record) for record in SeqIO.parse(file1, "fasta"))
output_file1 = file1.replace(".fa", "_modified.fa")
SeqIO.write(records1, output_file1, "fasta")
print(f"Updated trimmed FASTA file saved as: {output_file1}")

# Read tree file and apply header modifications
with open(tree_file, "r") as f:
    tree_data = f.read()

# Replace old headers with modified headers in the tree
for original, modified in header_mapping.items():
    tree_data = tree_data.replace(original, modified)

# Save modified tree file
output_tree_file = tree_file.replace(".contree", "_modified.contree")
with open(output_tree_file, "w") as f:
    f.write(tree_data)

print(f"Updated tree file saved as: {output_tree_file}")

In [3]:
# after tree vis, can filter out non-rfl clade entirely
# rest of rfls and newly id hits are justifiable
def df_to_fasta(df, output_file):
    with open(output_file, 'w') as f:
        for _, row in df.iterrows():
            f.write(f"{row['Headers']}\n{row['Sequences']}\n")

In [ ]:
file_large = '/home/pmh3ax/pprs_analysis_other/rfl-tree/Sl5trim_rfl_seqs_modified.fa'
file_small = '/home/pmh3ax/pprs_analysis_other/Sl5trim_rfl_seqs.fa-extract.fasta'
dfl = fasta_df(file_large)
dfs = fasta_df(file_small)

headers_to_remove = dfs['Headers'].tolist()
dfl_filtered = dfl[~dfl['Headers'].isin(headers_to_remove)]

dfl_filtered['Headers'] = dfl_filtered['Headers'].str.replace(r'(.*RFL_)', r'>', regex=True)

output_file = '/home/pmh3ax/pprs_analysis_other/Sl5trimmed_rfl_seqs.fasta'
df_to_fasta(dfl_filtered, output_file)

In [ ]:
file_large = '/home/pmh3ax/pprs_analysis_other/Sl5allpprsseqs.fa'
file_small = '/home/pmh3ax/pprs_analysis_other/Sl5trimmed_rfl_seqs.fasta'

dfl = fasta_df(file_large)
dfs = fasta_df(file_small)

dfl['Headers'] = dfl['Headers'].str.strip()
dfs['Headers'] = dfs['Headers'].str.strip().str.lstrip('>')

headers_to_keep = set(dfs['Headers'])
dfl_filtered = dfl[dfl['Headers'].str.contains('|'.join(headers_to_keep), na=False)]

output_file = '/home/pmh3ax/pprs_analysis_other/Sl5update_rfl_seqs.fasta'
df_to_fasta(dfl_filtered, output_file)

In [ ]:
# run fasta df
# pprs fasta result
fasta = fasta_df('/home/pmh3ax/pprs_analysis_other/Sl5update_rfl_seqs.fasta')
fasta
# line below gets rid > at beginning, only run once
fasta['Headers'] = fasta['Headers'].str.slice(start=1)
string_list = fasta['Headers'].tolist()

In [ ]:
# gff to match, will work with pprfinder output since the ID and type columns contain matching IDs
gff = pd.read_csv('/home/pmh3ax/pprs_analysis_other/Sl5_pprs_id_allpprs.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
gff

In [ ]:
# match gff annotations with pprs defined by pprfinder output
# False, non ppr = 0
# True, ppr = 1
gff['PPR'] = gff['ID'].apply(lambda x: any(header in x for header in string_list))
gffppr = gff[gff['PPR']==True]
gffppr.drop('PPR', axis=1)

In [ ]:
len(gffppr)

In [ ]:
# save to gff
gffppr.to_csv('/home/pmh3ax/pprs_analysis_other/Sl5realign_rfl.gff', sep='\t', index=False, header=None)

In [ ]:
# get gff, run realign rfl, run realign all then region for all, blast
# don't forget to trim when realigning (after geneious mafft local)

In [9]:
# checking here
# compare rflctd-id pprs with new id pprs
fa_rfl = fasta_df('/home/pmh3ax/pprs_analysis_other/Sl5rflseqs.fa')
fa_new = fasta_df('/home/pmh3ax/pprs_analysis_other/Sl5update_rfl_seqs.fasta')

In [10]:
# name
fa_new['rflctdHeaders'] = fa_new['Headers']
fa_new['rflctdHeaders'] = fa_new['Headers'].apply(lambda x: str(x) + 'rfl' if x in fa_rfl['Headers'].values else x)

In [11]:
# count_rfl = fa_new['rflctdHeaders'].str.contains('rfl').sum()
# count_rfl
fa_new

,Headers,Sequences,rflctdHeaders
0,>_chr12_000315,GAGCATATATAGCCGACTGCTGACCAAGTAGTGTAGTTACAACTTA...,>_chr12_000315
1,>_chr12_004846,AATTGTTGTATCGCATACCCTACTCTTCAGTCTTCAGTTTCAATTG...,>_chr12_004846
2,>_chr12_004838,TTAGCCGACAAACTCAGCCGACAAACTCACTTCATTTCACTCCTGC...,>_chr12_004838
3,>_chr12_000323,CGGTCTCAATGACATTCCTAGCAAGCACGTAGGCTATCAAGTGCAC...,>_chr12_000323
4,>_chr12_004821,ATCAAACTGAAGCTGACTAAACCCTACTTCTCCTTTCCATTTCTTC...,>_chr12_004821
5,>_chr12_000376,AGACAAAGAAGCATGCTTGTTTTATTTTGTGTGGGACTGTGGGTAA...,>_chr12_000376
6,>_chr12_003042,TCTTCATCCCCGCTCCTAGTAACAACCATCGTCATCAACGAACTTC...,>_chr12_003042
7,>_chr12_002307,AGTACGAGATTGGCGGGAGAAGAACTATTAATTCGGCAACTCCATG...,>_chr12_002307
8,>_chr12_002922,AGTTAGATACCGCATGCTGCGCACGCGCATTTCTAATGTATTTGAT...,>_chr12_002922
9,>_chr12_002489,TATCATCACTTCATTATCGCTCTGCAAATTGCTCGCCATTTCTGCA...,>_chr12_002489


In [12]:
# Drop the existing 'Headers' column
fa_new.drop(columns=['Headers'], inplace=True)
# Move 'rflctdHeaders' to the first column and rename it to 'Headers'
fa_new.insert(0, 'Headers', fa_new.pop('rflctdHeaders'))
fa_new

,Headers,Sequences
0,>_chr12_000315,GAGCATATATAGCCGACTGCTGACCAAGTAGTGTAGTTACAACTTA...
1,>_chr12_004846,AATTGTTGTATCGCATACCCTACTCTTCAGTCTTCAGTTTCAATTG...
2,>_chr12_004838,TTAGCCGACAAACTCAGCCGACAAACTCACTTCATTTCACTCCTGC...
3,>_chr12_000323,CGGTCTCAATGACATTCCTAGCAAGCACGTAGGCTATCAAGTGCAC...
4,>_chr12_004821,ATCAAACTGAAGCTGACTAAACCCTACTTCTCCTTTCCATTTCTTC...
5,>_chr12_000376,AGACAAAGAAGCATGCTTGTTTTATTTTGTGTGGGACTGTGGGTAA...
6,>_chr12_003042,TCTTCATCCCCGCTCCTAGTAACAACCATCGTCATCAACGAACTTC...
7,>_chr12_002307,AGTACGAGATTGGCGGGAGAAGAACTATTAATTCGGCAACTCCATG...
8,>_chr12_002922,AGTTAGATACCGCATGCTGCGCACGCGCATTTCTAATGTATTTGAT...
9,>_chr12_002489,TATCATCACTTCATTATCGCTCTGCAAATTGCTCGCCATTTCTGCA...


In [13]:
df_to_fasta(fa_new, '/home/pmh3ax/pprs_analysis_other/Sl5marked_rfl_seqs.fasta')

In [14]:
align = fasta_df('/home/pmh3ax/pprs_analysis_other/rfl-tree/Sl5updatetrim_rfl_seqs.fa')

In [15]:
# Assuming fa_new has headers in a column named 'header' and align has sequences in a column named 'sequence'
merged_df = fa_new[['Headers']].join(align[['Sequences']])
merged_df

,Headers,Sequences
0,>_chr12_000315,ctcac--------------aagtcacaagtcacaaccctactttca...
1,>_chr12_004846,----------------------------------------------...
2,>_chr12_004838,------------ttagccgacaaactcagccgacaaactcacttca...
3,>_chr12_000323,--------cggtctcaatgacattcctagcaagcacgtaggctatc...
4,>_chr12_004821,-----------------------------------------cccta...
5,>_chr12_000376,cccagatagtcacacaatcacaatc--------------------a...
6,>_chr12_003042,gtcataactcgcaatgaactaaagtttataaaacatttgaatccga...
7,>_chr12_002307,----------------------------------------------...
8,>_chr12_002922,gatacgctcgcacgcgcatttctaatgtatttgatctccttttttt...
9,>_chr12_002489,---------------------------------tatcatcacttca...


In [16]:
count_rfl = merged_df['Headers'].str.contains('rfl').sum()
count_rfl

6

In [17]:
df_to_fasta(merged_df, '/home/pmh3ax/pprs_analysis_other/Sl5marked_rfl_seqs_align.fasta')

In [ ]:
# copy for un-marking for alignment, tree again

In [21]:
# checking here
# compare rflctd-id pprs with new id pprs
fa_mark = fasta_df('/home/pmh3ax/pprs_analysis_other/Sl5marked_rfl_seqs_align_mod.fasta')

In [22]:
# name
fa_mark['rflctdHeaders'] = fa_mark['Headers']
fa_mark['rflctdHeaders'] = fa_mark['Headers'].apply(lambda x: x.replace('rfl', '') if 'rfl' in x else x)

In [23]:
#count_rfl = fa_mark['rflctdHeaders'].str.contains('rfl').sum()
#count_rfl
fa_mark = fa_mark.iloc[1:]
fa_mark

,Headers,Sequences,rflctdHeaders
1,>_chr12_004846,----------------------------------------------...,>_chr12_004846
2,>_chr12_004838,------------TTAGCCGACAAACTCAGCCGACAAACTCACTTCA...,>_chr12_004838
3,>_chr12_000323,--------CGGTCTCAATGACATTCCTAGCAAGCACGTAGGCTATC...,>_chr12_000323
4,>_chr12_004821,-----------------------------------------CCCTA...,>_chr12_004821
5,>_chr12_000376,CCCAGATAGTCACACAATCACAATC--------------------A...,>_chr12_000376
6,>_chr12_003042,GTCATAACTCGCAATGAACTAAAGTTTATAAAACATTTGAATCCGA...,>_chr12_003042
7,>_chr12_002307,----------------------------------------------...,>_chr12_002307
8,>_chr12_002922,GATACGCTCGCACGCGCATTTCTAATGTATTTGATCTCCTTTTTTT...,>_chr12_002922
9,>_chr12_002489,---------------------------------TATCATCACTTCA...,>_chr12_002489
10,>_scaffold_13_000458,----------------------------------------------...,>_scaffold_13_000458


In [24]:
# Drop the existing 'Headers' column
fa_mark.drop(columns=['Headers'], inplace=True)
# Move 'rflctdHeaders' to the first column and rename it to 'Headers'
fa_mark.insert(0, 'Headers', fa_mark.pop('rflctdHeaders'))
fa_mark

,Headers,Sequences
1,>_chr12_004846,----------------------------------------------...
2,>_chr12_004838,------------TTAGCCGACAAACTCAGCCGACAAACTCACTTCA...
3,>_chr12_000323,--------CGGTCTCAATGACATTCCTAGCAAGCACGTAGGCTATC...
4,>_chr12_004821,-----------------------------------------CCCTA...
5,>_chr12_000376,CCCAGATAGTCACACAATCACAATC--------------------A...
6,>_chr12_003042,GTCATAACTCGCAATGAACTAAAGTTTATAAAACATTTGAATCCGA...
7,>_chr12_002307,----------------------------------------------...
8,>_chr12_002922,GATACGCTCGCACGCGCATTTCTAATGTATTTGATCTCCTTTTTTT...
9,>_chr12_002489,---------------------------------TATCATCACTTCA...
10,>_scaffold_13_000458,----------------------------------------------...


In [26]:
df_to_fasta(fa_mark, '/home/pmh3ax/pprs_analysis/Sl5_last_run_rfl_seqs.fasta')

In [47]:
align = fasta_df('/home/pmh3ax/pprs_analysis/realign_rfl_seqs.fa')

In [53]:
# Assuming fa_new has headers in a column named 'header' and align has sequences in a column named 'sequence'
merged_df = fa_new[['Headers']].join(align[['Sequences']])
merged_df

,Headers,Sequences
0,>Silene_vulgaris_scaffold_1_003467.1,----------------------------------------------...
1,>Silene_vulgaris_scaffold_1_002736.1,----------------------------------------------...
2,>Silene_vulgaris_scaffold_2_003837.1,----------------------------------------------...
3,>Silene_vulgaris_scaffold_5_FUN_025820-T1,----------------------------------------------...
4,>Silene_vulgaris_scaffold_5_001830.1,----------------------------------------------...
...,...,...
151,>Silene_vulgaris_scaffold_13_FUN_054347-T1,----------------------------------------------...
152,>Silene_vulgaris_scaffold_16_FUN_068604-T1,----------------------------------------------...
153,>Silene_vulgaris_scaffold_16_FUN_068798-T1,tgcgtgattttagaaacattagttgcctttttgat-----------...
154,>Silene_vulgaris_scaffold_16_FUN_068800-T1,----------------------------------------------...


In [54]:
count_rfl = merged_df['Headers'].str.contains('rfl').sum()
count_rfl

98

In [55]:
df_to_fasta(merged_df, '/home/pmh3ax/pprs_analysis/marked_rfl_seqs_align.fasta')

In [7]:
# run fasta df
# pprs fasta result
fasta = fasta_df('/home/pmh3ax/pprs_analysis/update_rfl_seqs.fasta')
fasta
# line below gets rid > at beginning, only run once
fasta['Headers'] = fasta['Headers'].str.slice(start=1)
string_list = fasta['Headers'].tolist()

In [ ]:
# gff to match, will work with pprfinder output since the ID and type columns contain matching IDs
gff = pd.read_csv('/home/pmh3ax/pprs_analysis/Sv3_pprs_id_allpprs.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type', 'Start', 'End', '.', 'Strand', 'score', 'ID']
gff

In [ ]:
# match gff annotations with pprs defined by pprfinder output
# False, non ppr = 0
# True, ppr = 1
gff['PPR'] = gff['ID'].apply(lambda x: any(header in x for header in string_list))
gffppr = gff[gff['PPR']==True]
gffppr.drop('PPR', axis=1)

In [ ]:
# save to gff- can now see on chr
gffppr.to_csv('/home/pmh3ax/pprs_analysis_other/Sc3realign_rfl.gff', sep='\t', index=False, header=None)

In [ ]:
"""
update for existing align, tree
tree in ete-blast
align all
update name
RETRIEVE gff with rfl or not, to id possible fakes...
"""

In [ ]:
# ignore below for now
# for next, will want to intersect to find ppr proteins for each ppr grouped
# then extract seqs for each specified ppr protein
# assess if will align better roughly, adjust gap penalty, align, check...
#

In [ ]:
# df to match, need to define fasta_df function before
vulg_motif = fasta_df('/home/pmh3ax/pprfinder_runs/treepipe/vulgaris/trim_vulgaris.fa')
vulg_motif['Headers'] = vulg_motif['Headers'].str.slice(start=1)

In [ ]:
def scaffold_header(fa_path, headers, loc_names, headers_new, output_path=None):
    # fasta file, headers column title, new fasta with updated names, new headers column title
    # initialize fasta
    fa = fasta_df(fa_path)
    # empty column for match
    fa['new']=''
    # init names
    loc = pd.read_csv(loc_names, sep='\t', header=None, names=['old_name', headers_new])
    loc_list = loc[headers_new].tolist()
    # Iterate through the FASTA DataFrame and update the headers
    for index, row in fa.iterrows():
        old_header = row[headers][1:]
        matching_new_header = next((new_header for new_header in loc_list if old_header in new_header), None)
        fa.at[index, 'new'] = matching_new_header if matching_new_header else old_header
    fa = fa.drop(headers, axis=1)
    fa = fa[['new','Sequences']]
    if output_path:
        fa.to_csv(output_path, sep='\t', index=False, header=None)
    return fa

In [ ]:
# function run- motif
scaffold_header('/home/pmh3ax/pprfinder_runs/treepipe(motif)/vulgaris/trim_vulgaris.fa', 'Headers',
                '/home/pmh3ax/pprfinder_runs/Sv3pprs_gene_scaf.gff', 'ID_clean',
                '/home/pmh3ax/pprfinder_runs/treepipe(motif)/vulgaris/name_vulgaris.fa')

In [ ]:
# function run- nt
scaffold_header('/home/pmh3ax/pprfinder_runs/treetranscript(nt)/vulgaris/trimt_vulgaris.fa', 'Headers',
                '/home/pmh3ax/pprfinder_runs/Sv3pprs_gene_scaf.gff', 'ID_clean',
                '/home/pmh3ax/pprfinder_runs/treetranscript(nt)/vulgaris/namet_vulgaris.fa')

In [ ]:
pprs_gene_filtered.to_csv('/home/pmh3ax/pprfinder_runs/v_run_2/Sv3pprs_gene.gff', sep='\t', index=False, header=None)

In [ ]:
# extract transcripts from pprs genes beads filtered
# cd gffread
./gffread -w /home/pmh3ax/pprfinder_runs/transcripts_vulgaris.fa -g /home/pmh3ax/start_files/S_vulgaris_v3.fasta /home/pmh3ax/pprfinder_runs/v_run_2/Sv3pprs_gene.gff

In [ ]:
# Filter beads to include only rows matching the filtered pprs_gene IDs
filtered_beads = beads[beads['ID'].isin(pprs_gene_filtered['ID_clean'])]
filtered_beads

# Extract the motif_clean column for the matching rows
motif_clean_filtered = filtered_beads['motif_clean']
filtered_beads

In [ ]:
# Specify the output FASTA file
output_fasta = "/home/pmh3ax/pprfinder_runs/motif_vulgaris.fa"

# Open the file in write mode
with open(output_fasta, 'w') as fasta_file:
    for _, row in filtered_beads.iterrows():
        # Write each ID_clean as the header and motif_clean as the sequence
        fasta_file.write(f">{row['ID']}\n{row['motif_clean']}\n")

print(f"FASTA file saved to {output_fasta}")

In [ ]:
# see jobs, mafft to align
# then fast tree and manual correct

In [ ]:

# please note these last few steps are still in refinement


In [ ]:
# run in command line
# using p which contains p (and pls), filter fastas to make 4 separate gene, exon, cds, mrna
grep -E 'gene' '/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_out.gff' > '/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_gene.gff'

In [ ]:
# starting gff
gff = pd.read_csv('/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_cds.gff', sep='\t', header=None)
gff.columns = ['chr', 'source', 'type_pprs', 'Start', 'End', 'score_pprs', 'Strand', 'frame', 'ID', 'length', 'motif_arrangement', 'type_beads', 'score_beads', 'motif_count']

In [ ]:
gff = gff.drop(columns=['length', 'motif_arrangement', 'type_beads', 'score_beads', 'motif_count'])

In [ ]:
updated_ids = []
for ID in gff['ID']:
    if ';' in ID:
        ID = ID.split(";")[0]
    if '=' in ID:
        ID = ID.split("=")[1]
    updated_ids.append(ID)
gff['ID'] = updated_ids
gff['type_pprs'] = gff['ID']

In [ ]:
# save to gff
gff.to_csv('/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_cds.gff', sep='\t', index=False, header=None)

In [ ]:
# run in command line- get fasta seqs
# update your version for module
module load bedtools
bedtools getfasta -nameOnly -fo '/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_cds.fasta' -fi '/home/pmh3ax/start_files/S_vulgaris_v3.fasta' -bed '/home/pmh3ax/pprfinder_runs/all_run/Sv3_pprsp_cds.gff'

In [ ]:
# cd hit to cluster and reduce sequence redundancy- don't use 
# command line
cdhit/bin/cd-hit -i pprfinder_runs/all_run/Sv3_pprs_exon.fasta -o pprfinder_runs/all_run/sv3_pprs_exon_hit

In [ ]:
# small break to get broken-down groups for better aligning

beadsp_onlys['motif_count'].value_counts().sort_index().plot.bar()

In [ ]:
def split_by_frequency_quartiles(df, column):
    # df (pd.DataFrame): The input DataFrame.
    # column (str): The column to calculate frequencies and quartiles.
    df = df.copy()
    df['quartile_group'] = pd.qcut(df[column], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    print(df[[column, 'frequency', 'quartile_group']])
    q1 = df[df['quartile_group'] == 'Q1']
    q2 = df[df['quartile_group'] == 'Q2']
    q3 = df[df['quartile_group'] == 'Q3']
    q4 = df[df['quartile_group'] == 'Q4']

    return q1, q2, q3, q4

beadsp_onlys1, beadsp_onlys2, beadsp_onlys3, beadsp_onlys4 = split_by_frequency_quartiles(beadsp_onlys, 'motif_count')

In [ ]:
beadsp_onlys1.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadsp_only1.gff', sep='\t', index=False, header=None)
beadsp_onlys2.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadsp_only2.gff', sep='\t', index=False, header=None)
beadsp_onlys3.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadsp_only3.gff', sep='\t', index=False, header=None)
beadsp_onlys4.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadsp_only4.gff', sep='\t', index=False, header=None)

In [ ]:
# small break to get broken-down groups for better aligning

beadspls_onlys['motif_count'].value_counts().sort_index().plot.bar()

In [ ]:
def split_by_frequency_quartiles(df, column):
    # df (pd.DataFrame): The input DataFrame.
    # column (str): The column to calculate frequencies and quartiles.
    df = df.copy()
    df['quartile_group'] = pd.qcut(df[column], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    print(df[[column,'quartile_group']])
    q1 = df[df['quartile_group'] == 'Q1']
    q2 = df[df['quartile_group'] == 'Q2']
    q3 = df[df['quartile_group'] == 'Q3']
    q4 = df[df['quartile_group'] == 'Q4']

    return q1, q2, q3, q4

beadspls_onlys1, beadspls_onlys2, beadspls_onlys3, beadspls_onlys4 = split_by_frequency_quartiles(beadspls_onlys, 'motif_count')

In [ ]:
beadspls_onlys1.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_only1.gff', sep='\t', index=False, header=None)
beadspls_onlys2.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_only2.gff', sep='\t', index=False, header=None)
beadspls_onlys3.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_only3.gff', sep='\t', index=False, header=None)
beadspls_onlys4.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_only4.gff', sep='\t', index=False, header=None)

In [ ]:
# small break to get broken-down groups for better aligning

beadspls_es['motif_count'].value_counts().sort_index().plot.bar()

In [ ]:
def split_by_frequency_quartiles(df, column):
    # df (pd.DataFrame): The input DataFrame.
    # column (str): The column to calculate frequencies and quartiles.
    df = df.copy()
    df['quartile_group'] = pd.qcut(df[column], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    print(df[[column,'quartile_group']])
    q1 = df[df['quartile_group'] == 'Q1']
    q2 = df[df['quartile_group'] == 'Q2']
    q3 = df[df['quartile_group'] == 'Q3']
    q4 = df[df['quartile_group'] == 'Q4']

    return q1, q2, q3, q4

beadspls_es1, beadspls_es2, beadspls_es3, beadspls_es4 = split_by_frequency_quartiles(beadspls_es, 'motif_count')

In [ ]:
beadspls_es1.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_e1.gff', sep='\t', index=False, header=None)
beadspls_es2.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_e2.gff', sep='\t', index=False, header=None)
beadspls_es3.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_e3.gff', sep='\t', index=False, header=None)
beadspls_es4.to_csv('/home/pmh3ax/pprfinder_runs/groups_by_motif/Sv3_beadspls_e4.gff', sep='\t', index=False, header=None)